# 3.3 Feature Engineering: all features against the reduced set #
Compares every model trained with all features (`../results/`) with the copies in `reduced_feature_set/`, which leave out the five features that `01_drop_column_ablation.ipynb` flagged. Nothing is retrained: the tables are built from the saved forecasts and business KPIs. The findings are in `README.md`.

| Report | Output |
|---|---|
| Table `tab:ablation-before-after` | `results/tables/validation_mae.tex` |
| Test MAE and bias, both feature sets (not in the report) | `results/tables/test_mae.tex` |
| Imbalance and cost, both feature sets (not in the report) | `results/tables/business_kpis.tex` |

**Inputs:** the results of `3_methods/03`–`06`, `4_business_evaluation/01_business_kpis.ipynb` and the same notebooks in `reduced_feature_set/`. **Run time:** a few seconds.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))   #helper modules shared by the notebooks
import pandas as pd
from forecast_checks import ForecastValidator, load_tables

#All features: the main results; reduced set: the copies without EXCLUDE_COLUMNS
RUNS = {"All features": {"models": Path("../results"),
                         "business": Path("../../4_business_evaluation/results/business_kpis.json")},
        "Reduced set": {"models": Path("reduced_feature_set/results"),
                        "business": Path("reduced_feature_set/results/business_kpis.json")}}
TABLES = Path("results/tables")
TABLES.mkdir(parents=True, exist_ok=True)
#Model: (file stem, name the forecast is saved under)
MODELS = {"XGBoost": ("xgboost", "XGBoost"), "CNN": ("cnn", "Neural network"), "LSTM": ("lstm", "LSTM"),
          "Local heads": ("cnn_local_heads", "Local heads")}
BASELINE = "Same hour last week"

In [2]:
def scores(suffix: str) -> pd.DataFrame:
    """MAE, RMSE, nMAE and bias of every model in both runs, over the hours in results/<stem><suffix>.json."""
    rows = {}
    for run, paths in RUNS.items():
        for model, (stem, saved_as) in MODELS.items():
            validator = ForecastValidator.load_predictions(paths["models"] / f"{stem}{suffix}.json")
            rows[(model, run)] = ForecastValidator.score(validator.actual, validator.forecasts[saved_as])
    return pd.DataFrame(rows).T.unstack().reindex(list(MODELS))   #columns: (metric, run)


def signed(x: float, digits: int = 3) -> str:
    #Minus sign in math mode, as in the report's tables
    return f"{x:.{digits}f}" if x >= 0 else f"$-${-x:.{digits}f}"


def change(new: float, old: float) -> str:
    return f"${100 * (new / old - 1):+.1f}$"


def write_table(name: str, caption: str, columns: str, header: list[str], rows: list[list[str]]) -> None:
    lines = [f"% Caption: {caption}", rf"\begin{{tabular}}{{{columns}}}", r"\toprule", *header, r"\midrule",
             *(" & ".join(row) + r" \\" for row in rows), r"\bottomrule", r"\end{tabular}", ""]
    (TABLES / name).write_text("\n".join(lines))

## Validation MAE (Table `tab:ablation-before-after`) ##
Pooled over the three validation windows.

In [3]:
validation = scores("_validation_predictions")["mae"]
write_table("validation_mae.tex",
            "Validation MAE before and after removing the five features flagged by the ablation study.", "lrrr",
            [r" & \multicolumn{3}{c}{Validation MAE [MW]} \\", r"\cmidrule(lr){2-4}",
             r"Model & All features & Reduced set & $\Delta$ [\%] \\"],
            [[model, f"{r['All features']:.3f}", f"{r['Reduced set']:.3f}", change(r["Reduced set"], r["All features"])]
             for model, r in validation.iterrows()])
validation.round(3)

,All features,Reduced set
XGBoost,0.625,0.591
CNN,0.353,0.383
LSTM,0.357,0.397
Local heads,0.410,0.435


## Test MAE and bias ##

In [4]:
test = scores("_predictions")
n_hours = len(ForecastValidator.load_predictions(RUNS["All features"]["models"] / "xgboost_predictions.json").actual)
write_table("test_mae.tex",
            "Test-set MAE and bias with all features and with the reduced feature set "
            f"(n = {n_hours:,} hours).".replace(",", " "), "lrrrrr",
            [r" & \multicolumn{3}{c}{Test MAE [MW]} & \multicolumn{2}{c}{Bias [MW]} \\",
             r"\cmidrule(lr){2-4} \cmidrule(lr){5-6}",
             r"Model & All features & Reduced set & $\Delta$ [\%] & All features & Reduced set \\"],
            [[model, f"{r['mae', 'All features']:.3f}", f"{r['mae', 'Reduced set']:.3f}",
              change(r["mae", "Reduced set"], r["mae", "All features"]),
              signed(r["bias", "All features"]), signed(r["bias", "Reduced set"])]
             for model, r in test.iterrows()])
test[["mae", "bias"]].round(3)

mae                     bias            
            All features Reduced set All features Reduced set
XGBoost            0.334       0.339        0.153       0.162
CNN                0.257       0.294       -0.002       0.180
LSTM               0.284       0.286        0.008      -0.047
Local heads        0.295       0.290       -0.181       0.150

## Total imbalance and cost exposure ##
From the business KPIs of both runs, all areas together.

In [5]:
overview = {run: load_tables(paths["business"])["overview"] for run, paths in RUNS.items()}
imbalance = pd.DataFrame({run: o.loc["Total abs. imbalance [MWh]"] for run, o in overview.items()})
cost = pd.DataFrame({run: o.loc["Imbalance cost exposure [EUR]"] for run, o in overview.items()})
write_table("business_kpis.tex",
            "Total absolute imbalance and imbalance cost exposure on the test period, all six price areas, "
            "with all features and with the reduced feature set.", "lrrrrr",
            [r" & \multicolumn{3}{c}{$I$ [MWh]} & \multicolumn{2}{c}{Cost [EUR]} \\",
             r"\cmidrule(lr){2-4} \cmidrule(lr){5-6}",
             r"Model & All features & Reduced set & $\Delta$ [\%] & All features & Reduced set \\"],
            [[f"{model} (baseline)" if model == BASELINE else model,
              f"{imbalance.loc[model, 'All features']:,.1f}", f"{imbalance.loc[model, 'Reduced set']:,.1f}",
              change(imbalance.loc[model, "Reduced set"], imbalance.loc[model, "All features"]),
              f"{cost.loc[model, 'All features']:,.0f}", f"{cost.loc[model, 'Reduced set']:,.0f}"]
             for model in imbalance.index])
pd.concat({"I [MWh]": imbalance, "Cost exposure [EUR]": cost}, axis=1).round(1)

I [MWh]             Cost exposure [EUR]            
                    All features Reduced set        All features Reduced set
Same hour last week       5192.1      5192.1             76759.8     76759.8
XGBoost                   3504.3      3553.4             50853.1     50611.4
CNN                       2697.4      3084.6             43747.1     54280.6
LSTM                      2979.0      3003.1             48137.0     44753.8
Local heads               3097.5      3041.8             47898.9     46771.0